# Lezione 7 — Il tuo chatbot come servizio web con Flask

## Obiettivi di apprendimento
- Capire l'architettura client-server di un'applicazione di IA generativa.
- Spiegare a cosa serve CORS e perché è necessario in un chatbot web.
- Riconoscere le rotte `GET /` e `POST /chatbot` di un server Flask reale.
- Avviare un server Flask dentro lo stesso notebook e testarlo con richieste HTTP reali.
- Identificare il limite dello storico di conversazione globale condiviso.

## Cosa costruiremo
Riprenderemo il chatbot Blenderbot della lezione 6 e lo esporremo come un servizio Flask reale
—le stesse rotte `GET /` e `POST /chatbot` del progetto guidato di IBM
(`LLM_application_chatbot-main/app.py`)— facendo girare il server in un thread all'interno di
questo stesso notebook, per poterlo testare con richieste HTTP senza bisogno di esporre una porta
verso internet.

## L'architettura client-server, in breve

- **Backend (Flask)** = la cucina: ha il modello caricato e sa generare le risposte.
- **Frontend (HTML/JS)** = il cameriere: porta il messaggio dell'utente e riporta la risposta.
- **CORS** = il permesso esplicito per accettare richieste da un'origine diversa da quella del server.

Ripassa l'analogia completa e il limite dello storico globale condiviso nel
[Capitolo 11 del manuale](../manual/11_chatbot_web_flask_IT.md).

## L'`app.py` reale del progetto guidato (riferimento)

Ecco il backend Flask completo del progetto `LLM_application_chatbot-main` di IBM (codice
reale). Non lo eseguiamo così com'è in Colab perché dipende da `templates/index.html`, ma è
esattamente quello che riprodurremo nelle celle di codice seguenti:

```python
from flask import Flask, request, render_template
from flask_cors import CORS
import json
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

app = Flask(__name__)
CORS(app)

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
conversation_history = []

@app.route('/', methods=['GET'])
def home():
    return render_template('index.html')

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    data = request.get_data(as_text=True)
    data = json.loads(data)
    input_text = data['prompt']
    history = "\n".join(conversation_history)
    inputs = tokenizer.encode_plus(history, input_text, return_tensors="pt")
    outputs = model.generate(**inputs)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    conversation_history.append(input_text)
    conversation_history.append(response)
    return response

if __name__ == '__main__':
    app.run()
```

E il frontend (`static/script.js`) lascia l'endpoint del chatbot non risolto, di proposito:

```javascript
const url = 'www.example.com';  // Make a POST request to this url
```

Risolverai proprio questo placeholder nella pratica di questa lezione.

## Come lo testiamo dentro Colab

Google Colab non ti permette di avviare con un clic un server Flask visibile pubblicamente, ma
puoi far girare Flask **dentro lo stesso processo Python**, in ascolto su `127.0.0.1`
(localhost), usando un thread (`threading.Thread`) affinché il server giri in parallelo senza
bloccare la cella. Poi gli facciamo richieste HTTP reali con la libreria `requests`, dallo
stesso notebook. È esattamente lo stesso server Flask del progetto reale, solo che non lo
esponiamo a internet.

In [ ]:
# Installiamo Flask e flask-cors (requests è già installato in Colab)
!pip install -q flask flask-cors requests

In [ ]:
# Riutilizziamo lo stesso chatbot Blenderbot della lezione 6
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Modello Blenderbot caricato.")

In [ ]:
import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)  # permette richieste da qualsiasi origine (vedi capitolo 11 del manuale)

# Storico della conversazione: un'unica lista globale (vedi il limite
# di design che spieghiamo nel capitolo 11 del manuale)
conversation_history = []

@app.route('/', methods=['GET'])
def home():
    # Nel progetto reale questo servirebbe templates/index.html;
    # qui restituiamo un testo semplice perché non abbiamo un frontend in Colab
    return "Server Flask del chatbot attivo. Usa POST /chatbot con {'prompt': '...'}"

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    data = request.get_json()
    input_text = data['prompt']

    # Come in app.py: uniamo lo storico in una stringa
    history = "\n".join(conversation_history)

    # Tokenizziamo storico + nuovo messaggio e generiamo la risposta
    inputs = tokenizer.encode_plus(history, input_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=60)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Salviamo lo scambio nello storico globale
    conversation_history.append(input_text)
    conversation_history.append(response)

    return jsonify({"response": response})

def run_flask_app():
    # use_reloader=False è importante: il ricaricatore automatico di Flask
    # non funziona bene dentro un thread di Colab
    app.run(port=5000, use_reloader=False)

# Lanciamo il server Flask in un thread in background, senza bloccare il notebook
server_thread = threading.Thread(target=run_flask_app, daemon=True)
server_thread.start()

time.sleep(3)  # diamo tempo al server di finire di avviarsi
print("Server Flask in esecuzione su http://127.0.0.1:5000")

In [ ]:
import requests

resp = requests.post(
    "http://127.0.0.1:5000/chatbot",
    json={"prompt": "Ciao, come stai?"}
)
print("Status:", resp.status_code)
print("Risposta:", resp.json())

In [ ]:
# Inviamo un secondo messaggio per vedere come lo storico globale si accumula
resp2 = requests.post(
    "http://127.0.0.1:5000/chatbot",
    json={"prompt": "Qual è il tuo colore preferito?"}
)
print("Risposta:", resp2.json())

# Lo storico globale ora contiene i due scambi precedenti.
# Se "un altro utente" chiamasse questo stesso server proprio ora, il suo
# messaggio si mescolerebbe nello stesso storico: questo è il limite di design
# reale che spieghiamo nel capitolo 11 del manuale.
print("\nStorico accumulato (condiviso da tutte le richieste):")
print(conversation_history)

## Riepilogo della lezione
- Un chatbot web separa il backend (Flask + modello) dal frontend (HTML/JS), comunicanti via HTTP.
- `flask_cors` abilita le richieste da altre origini.
- La rotta `POST /chatbot` riceve il prompt, genera la risposta e aggiorna lo storico.
- In Colab, abbiamo testato il server reale facendolo girare in un thread su `127.0.0.1` e chiamandolo con `requests`, senza bisogno di esporlo a internet.
- Lo storico di conversazione è una lista globale condivisa da tutte le richieste: un limite reale di cui bisogna tenere conto.

Continua con la [pratica di questa lezione](../notebooks_practica/clase07_practica_chatbot_web_flask_IT.ipynb)
e ripassa il [Capitolo 11 del manuale](../manual/11_chatbot_web_flask_IT.md) se vuoi approfondire.